# Курсовая работа "Обработка сигналов с робота в реальных условиях"

## Общая информация

### Метки для работы с ноутбуком ##

* 1. **Тезис** — короткая формулировка, зачем нужен этот шаг.
* 2. **Объяснение** — что физически или математически происходит.
* 3. **Код** — воспроизводимая реализация.
* 4. **Комментарий** — что видно на графиках/таблицах.

### Робот и полигон NCLT



#### 1. Общие сведения

Для анализа используются данные **North Campus Long-Term (NCLT)** — крупномасштабного долговременного датасета для робототехники, собранного на роботизированной платформе **Segway** в кампусе University of Michigan, North Campus.

Датасет собирался примерно раз в две недели в течение **15 месяцев** (с января 2012 по апрель 2013), всего **27 сессий**. Суммарно: около **34.9 часов** логов и **147.4 км** траектории робота. Каждая сессия покрывает примерно одну и ту же территорию кампуса, но траектории, время суток и условия меняются. Съёмка велась как **внутри помещений**, так и **на улице**: коридоры и здания, дороги, тротуары, пешеходные дорожки.

Особенность датасета — повторное посещение одной и той же среды при разных условиях:

- движущиеся объекты: пешеходы, велосипедисты, автомобили;
- изменение освещения в течение дня;
- сезонные изменения: листва, снег, погода;
- долговременные структурные изменения: стройки, перестановка мебели внутри помещений;
- разные точки зрения и разные траектории проезда.

Для всех сессий предоставляется **ground-truth поза робота в единой системе координат**, полученная SLAM-ом с использованием лидарного scan matching и RTK GPS.

#### 2. Платформа и сенсоры

Робот — это **Segway robotic platform**, на которую установлены сенсоры восприятия, навигационные датчики и бортовой компьютер. Все данные логируются и получают метки времени на одном компьютере, установленном на Segway.

##### 2.1. Сенсоры восприятия

| Сенсор | Модель | Что даёт | Ключевые параметры |
|---|---|---|---|
| 3D-лидар | **Velodyne HDL-32E** | 3D-облако точек, \(360^\circ\) по азимуту | 32 лазера, дальность до 100 м, вращение 10 Гц; установлен «вверх ногами» для лучшего обзора высоких объектов |
| Омнинаправленная камера | **PointGrey Ladybug3** | панорамные изображения, видео | 6 камер по 2 Мп (\(1600\times1200\)): 5 в горизонтальном кольце + 1 вертикально; покрытие более 80% сферы; global shutter; автоэкспозиция; съёмка на 5 Гц |
| Планарный лидар | **Hokuyo UTM-30LX** | 2D-скан в горизонтальной плоскости | один луч, дальность 30 м, поле зрения \(270^\circ\); установлен горизонтально спереди |
| Планарный лидар | **Hokuyo URG-04LX** | 2D-скан «щёткой» вперёд, для плоскости земли | один луч, дальность 4 м, поле зрения \(240^\circ\); установлен в push-broom конфигурации |

Изображения Ladybug3 в архиве датасета хранятся как TIFF-файлы по камерам `Cam0`–`Cam5`; в тексте статьи также указано JPEG-сжатие при сборе, поэтому фактический формат в конкретном архиве стоит проверить перед чтением.

##### 2.2. Навигационные сенсоры

| Сенсор | Модель | Что даёт | Ключевые параметры |
|---|---|---|---|
| IMU | **Microstrain 3DM-GX3-45** | 3-осевые акселерометры, гироскопы, магнитометры; внутренний фильтр даёт 3D position, velocity, attitude | выход фильтрованных данных на 100 Гц |
| Одноосный гироскоп | **KVH DSP-3000 FOG** | высокоточное измерение вращения; используется для интегрированного yaw | волоконно-оптический гироскоп, одноосный |
| Consumer GPS | **Garmin 18x** | обычный GPS | 5 Гц |
| RTK GPS | **NovAtel DL-4 plus** | высокоточный GPS с RTK-поправками | 1 Гц; на кампусе установлена RTK-базовая станция |

Для нашего анализа особый интерес представляют:

- **IMU**: `ms25.csv` — магнитное поле, ускорение, угловые скорости; `ms25_euler.csv` — углы Эйлера roll, pitch, heading;
- **гироскоп**: угловые скорости \(p,q,r\) из IMU и интегрированный yaw из FOG, который используется в одометрии;
- **одометр**: `odometry_mu_100hz.csv` и `odometry_cov_100hz.csv` — оценка одометрии на 100 Гц; `odometry_mu.csv` и `odometry_cov.csv` — одометрия, синхронизированная с событиями изображений.

#### 3. Системы координат

В датасете используется 6-DOF представление позы:

\[
\mathbf{x}_{ij} = \left[x_{ij}, y_{ij}, z_{ij}, \phi_{ij}, \theta_{ij}, \psi_{ij}\right]^\top,
\]

где \(\phi\) — roll вокруг \(x\), \(\theta\) — pitch вокруг \(y\), \(\psi\) — yaw вокруг \(z\). Порядок поворотов: \(\mathrm{rotz}(\psi) \rightarrow \mathrm{roty}(\theta) \rightarrow \mathrm{rotx}(\phi)\).

##### 3.1. Body frame робота

Система координат корпуса Segway:

- начало — на оси между колёсами;
- \(x\) — вперёд;
- \(y\) — вправо;
- \(z\) — вниз.

Все сенсоры имеют свои фреймы, заданные относительно body frame. Соответствующие 6-DOF трансформации приведены в статье в Table 4; их можно использовать для пересчёта измерений в корпус робота.

##### 3.2. Локальная система координат

Позы робота в датасете задаются в локальной системе, привязанной к сторонам света:

- \(x\) — север;
- \(y\) — восток;
- \(z\) — вниз.

Начало локальной системы фиксировано в GPS-координатах:

| Параметр | Значение |
|---|---|
| Широта | \(42.293227^\circ\) |
| Долгота | \(-83.709657^\circ\) |
| Высота | 270 м |
| Экваториальный радиус Земли \(r_e\) | 6 378 135 м |
| Полярный радиус \(r_p\) | 6 356 750 м |

Пересчёт GPS \(\leftrightarrow\) локальные координаты выполняется линеаризацией вокруг этого начала с локальными радиусами \(r_{ns}\) и \(r_{ew}\). Для наших задач это важно, потому что одометрия и ground-truth заданы в локальном фрейме, а GPS — в широте/долготе.

#### 4. Одометрия и ground-truth

Одометрия в NCLT построена на **EKF** с differential-drive моделью процесса. Входы:

- скорости левого и правого колёс \(v_l, v_r\);
- изменение курса \(\delta_\psi\) по одноосному FOG;
- обновления по roll, pitch и body-frame angular rates от IMU.

Состояние:

$$
\mathbf{x}_t = [x, y, \phi, \theta, \psi, p, q, r]^\top.
$$

Высота \(z\) в одометрии не оценивается, потому что изменение \(z\) не наблюдаемо по сенсорам одометрии Segway. Высота восстанавливается в ground-truth за счёт RTK GPS и лидарного scan matching.

В датасете предоставлены две формы одометрии:

1. **100 Гц**: среднее и ковариация оценки одометрии в локальном фрейме;
2. **относительная одометрия между узлами**, синхронизированными с изображениями и предварительно извлечёнными лидарными сканами — удобно для построения SLAM-графа.

Ground-truth построен как большой SLAM-решение по всем сессиям:

- внутрисессионные и межсессионные лидарные scan matching-факторы;
- RTK GPS, где он доступен и надёжен;
- автоматическое добавление scan matching между парами узлов в пределах 8 м;
- ручное удаление плохих факторов;
- интерполяция поз между узлами графа по одометрии.

Отдельно предоставляется файл `gps_rtk_err.csv` — ошибка RTK GPS относительно ground-truth. Это полезно, потому что рядом со зданиями RTK может давать ошибки более 10 м из-за multipath, а внутри помещений RTK часто недоступен.

#### 5. Что это значит для анализа сигналов

Рассматриваемы величины:

- угловые скорости \(p,q,r\) и ускорения из IMU;
- углы Эйлера roll, pitch, heading;
- одометрические \(x,y,\psi\) и их ковариации;
- при необходимости — yaw/курс и скорости движения.

Перед спектральным анализом нужно учитывать:

- метки времени заданы как **UTIME в микросекундах** от UNIX epoch;
- сигналы могут быть нерегулярными по времени, поэтому перед FFT нужен ресемплинг на равномерную сетку, например 100 Гц;
- единицы: угловые скорости — rad/s, ускорения — м/с², одометрия \(x,y,z\) — м, углы — rad;
- для гироскопа важно разделять «сырые» угловые скорости IMU и интегрированный yaw FOG, который входит в одометрию;
- полезно вычитать среднее/тренд и проверять пропуски, дубликаты и выбросы.

### План работы (по каждому датчику)



 1. Загрузить данные, восстановить реальную частоту дискретизации.

 2. Посмотреть на сырые сигналы, оценить масштаб и характер.

 3. Убрать смещение нуля (bias) и линейный тренд — иначе низкочастотная
    составляющая замаскирует полезные колебания.

 4. Построить амплитудные спектры (FFT), найти характерные частоты.

 5. Построить спектрограммы (STFT), посмотреть, как частоты меняются
    во времени.

 6. Найти аномалии двумя независимыми методами: по \(z\)-оценке и по
    скользящей дисперсии.

 7. Проинтегрировать угловые скорости в углы ориентации и обсудить
    накопление дрейфа.
    
 8. Сопоставить аномалии по трём осям — понять, какие события физические,
    а какие — шум отдельной оси.

### Общие тезисы

* Используем **сырой** сигнал, без внутренней фильтрации IMU.
* Работаем с угловыми скоростями \(gx, gy, gz\) в body frame робота.
* Сессия `2013-01-10` — зимняя (Snow: Yes), значит, сигнал «грязный»,
    это полезно для демонстрации методов.
* Частота дискретизации — порядка 100 Гц; точное значение получим из
    данных.

### Импорты, загрузка файлов и настройки

In [1]:
!pip install pathlib
!pip install matplotlib
!pip install numpy
!pip install pandas
!pip install scipy


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Artem\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Artem\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Artem\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Artem\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Artem\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import detrend, spectrogram, stft
from scipy.integrate import cumulative_trapezoid

from __future__ import annotations

import argparse
import shutil
import sys
import tarfile
import urllib.error
import urllib.request
from pathlib import Path

In [4]:
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

### Общие настройки для графиков

In [5]:
# --- Стиль графиков: одинаковый во всех ячейках ---
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
    "axes.titlesize": 11,
    "figure.autolayout": False,
})

# --- Единый набор цветов для осей (используется везде ниже) ---
AXIS_COLORS = {
    "gx": "tab:red",
    "gy": "tab:green",
    "gz": "tab:blue",
}
AXIS_NAMES = {
    "gx": "Roll (ось X)",
    "gy": "Pitch (ось Y)",
    "gz": "Yaw (ось Z)",
}


## Гироскоп ##

### Общая информация про гироскоп

... Теория ...

### Загрузка данных с сайта ???

In [ ]:
BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data"
DEFAULT_DATE = "2013-01-10"
NCLT_HOSTS = [
    "https://s3.us-east-2.amazonaws.com/nclt.perl.engin.umich.edu/sensor_data",
]
ARCHIVE_TEMPLATE = "{date}_sen.tar.gz"
REQUIRED_CSV = Path("sen") / "ms25.csv"


def archive_path(date: str) -> Path:
    return DATA_DIR / ARCHIVE_TEMPLATE.format(date=date)


def csv_path() -> Path:
    return DATA_DIR / REQUIRED_CSV


def build_urls(date: str) -> list[str]:
    name = ARCHIVE_TEMPLATE.format(date=date)
    return [f"{host}/{name}" for host in NCLT_HOSTS]


def manual_instructions(date: str) -> str:
    name = ARCHIVE_TEMPLATE.format(date=date)
    urls = "\n".join(f"  - {u}" for u in build_urls(date))
    return (
        "\n" + "=" * 78 + "\n"
        "НЕ УДАЛОСЬ СКАЧАТЬ АРХИВ АВТОМАТИЧЕСКИ.\n"
        "\nВыполните следующие шаги вручную:\n"
        f"  1. Откройте сайт NCLT: https://robots.engin.umich.edu/nclt/\n"
        f"  2. Скачайте архив сенсорных данных за {date}: {name}\n"
        "     Прямые ссылки (одна из них):\n"
        f"{urls}\n"
        f"  3. Положите файл в папку: {DATA_DIR}\n"
        "  4. Повторно запустите: python download_data.py\n"
        "     Скрипт распакует уже скачанный архив.\n"
        + "=" * 78 + "\n"
    )


def download(date: str, force: bool = False) -> Path:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    target = archive_path(date)
    if target.exists() and not force:
        print(f"[i] Архив уже существует: {target}")
        return target

    last_error: Exception | None = None
    for url in build_urls(date):
        print(f"[i] Скачивание: {url}")
        try:
            with urllib.request.urlopen(url, timeout=60) as response, open(
                target, "wb"
            ) as out:
                shutil.copyfileobj(response, out)
            print(f"[+] Сохранено: {target} ({target.stat().st_size / 1e6:.1f} MB)")
            return target
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError) as exc:
            last_error = exc
            print(f"[!] Ошибка при скачивании {url}: {exc}")
            if target.exists():
                target.unlink()

    raise RuntimeError(
        f"Не удалось скачать {ARCHIVE_TEMPLATE.format(date=date)}. "
        f"Последняя ошибка: {last_error}"
    )


def extract(archive: Path) -> Path:
    if not archive.exists():
        raise FileNotFoundError(f"Архив не найден: {archive}")
    print(f"[i] Распаковка {archive.name} в {DATA_DIR} ...")
    with tarfile.open(archive, "r:gz") as tar:
        members = []
        for member in tar.getmembers():
            member_path = (DATA_DIR / member.name).resolve()
            if not str(member_path).startswith(str(DATA_DIR.resolve())):
                raise RuntimeError(f"Небезопасный путь в архиве: {member.name}")
            members.append(member)
        tar.extractall(DATA_DIR, members=members)

    candidates = [csv_path()] + sorted(DATA_DIR.rglob("ms25.csv"))
    for candidate in candidates:
        if candidate.exists():
            if candidate != csv_path():
                csv_path().parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(candidate, csv_path())
            print(f"[+] Найден файл сырых данных: {csv_path()}")
            return csv_path()

    raise FileNotFoundError(
        "ms25.csv не найден после распаковки. "
        f"Проверьте содержимое папки {DATA_DIR}."
    )


def main() -> int:
    parser = argparse.ArgumentParser(description="Загрузка данных NCLT")
    parser.add_argument("--date", default=DEFAULT_DATE, help="Дата (YYYY-MM-DD)")
    parser.add_argument("--force", action="store_true", help="Скачать заново")
    args = parser.parse_args()

    # 1. Скачиваем данные датчиков (если нет)
    if not csv_path().exists() or args.force:
        try:
            archive = download(args.date, force=args.force)
            extract(archive)
        except Exception as exc:
            print(f"[!] Ошибка с сенсорами: {exc}")
            print(manual_instructions(args.date))
            return 1
    else:
        print(f"[i] Файл датчиков уже существует: {csv_path()}")

    # 2. Скачиваем Ground Truth (если нет)
    gt_file = DATA_DIR / f"{args.date}_groundtruth.csv"
    if not gt_file.exists() or args.force:
        gt_url = f"https://s3.us-east-2.amazonaws.com/nclt.perl.engin.umich.edu/groundtruth/{args.date}_groundtruth.csv"
        print(f"[i] Скачивание Ground Truth: {gt_url}")
        try:
            import urllib.request
            urllib.request.urlretrieve(gt_url, gt_file)
            print(f"[+] Ground Truth сохранен: {gt_file}")
        except Exception as e:
            print(f"[!] Не удалось скачать Ground Truth: {e}")
    else:
        print(f"[i] Ground Truth уже существует: {gt_file}")

    print("[+] Готово. Данные готовы к анализу.")
    return 0


if __name__ == "__main__":
    sys.exit(main())

### Инициализация данных

In [ ]:
DATA = Path("")
PLOTS = Path("plots")
PLOTS.mkdir(exist_ok=True)

COLS = [
    "t_us",
    "mag_x", "mag_y", "mag_z",
    "acc_x", "acc_y", "acc_z",
    "gx", "gy", "gz",
]
df = pd.read_csv(DATA, header=None, names=COLS)

### Реальное время 

... Теория ...

In [ ]:
# Время в секундах от начала записи
t = (df["t_us"].to_numpy(dtype=np.float64) - float(df["t_us"].iloc[0])) / 1e6

# Реальная частота дискретизации (медиана устойчива к пропускам)
dt = np.diff(t)
fs = 1.0 / float(np.median(dt))
duration = float(t[-1] - t[0])

print(f"Отсчётов: {len(t)} | fs ≈ {fs:.2f} Гц | Длительность: {duration:.1f} с")

### Функции для обработки и визуализации

#### Очистка сигнала, поиск аномалий. Возвращает очищенный сигнал, параметры и DataFrame.

... Теория ...

In [ ]:
def analyze_axis(raw_sig: np.ndarray, t: np.ndarray, axis_name: str, threshold: float = 3.5):
    # 1. Убираем bias и линейный тренд
    bias = float(raw_sig.mean())
    clean_sig = detrend(raw_sig - bias, type="linear")

    # 2. Считаем СКО и Z-оценки
    sigma = float(clean_sig.std(ddof=1))
    z_scores = clean_sig / sigma

    # 3. Ищем аномалии
    mask = np.abs(z_scores) > threshold
    anomalies = pd.DataFrame({
        "t": t[mask],
        "raw": raw_sig[mask],
        "detrended": clean_sig[mask],
        "z": z_scores[mask],
        "level": np.abs(z_scores[mask])
    })

    return clean_sig, bias, sigma, mask, anomalies

### Визуализация

#### График очищенного сигнала

In [ ]:
def plot_time_domain(t: np.ndarray, clean_sig: np.ndarray, mask: np.ndarray,
                     sigma: float, display_name: str, color: str, filename: str):
    """Строит график очищенного сигнала и аномалий."""
    fig, ax = plt.subplots(figsize=(12, 4))

    # Только очищенный сигнал (сырой убран, чтобы не перекрывать график)
    ax.plot(t, clean_sig, lw=0.4, color=color, label="Очищенный сигнал")

    # Коридор ±3.5 сигм
    ax.axhline(+3.5 * sigma, ls="--", color="tab:orange", lw=1.2, label="Порог ±3.5σ")
    ax.axhline(-3.5 * sigma, ls="--", color="tab:orange", lw=1.2)

    # Точки аномалий
    ax.scatter(t[mask], clean_sig[mask], s=20, color="tab:red", zorder=3,
               label=f"Аномалии ({mask.sum()})")

    ax.set_xlabel("Время, с")
    ax.set_ylabel(f"{display_name}, рад/с")
    ax.set_title(f"Гироскоп ({display_name}): очищенный сигнал и аномалии")
    ax.legend(loc="upper right", fontsize=9)

    fig.tight_layout()
    fig.savefig(PLOTS / filename)
    plt.close(fig)

#### Линейный спектр с окном Ханна

`Линейный спектр` - ...


`Окно Ханна` - ...

In [ ]:
def plot_spectrum(clean_sig: np.ndarray, fs: float, display_name: str, color: str, filename: str):
    """Строит спектр (линейный и в дБ) с окном Ханна."""
    N = len(clean_sig)
    window = np.hanning(N)
    spectrum = np.fft.rfft(clean_sig * window)
    freqs = np.fft.rfftfreq(N, d=1.0 / fs)

    # Амплитуда с учётом когерентного усиления окна
    amplitude = np.abs(spectrum) / float(window.sum())
    amplitude_db = 20.0 * np.log10(np.maximum(amplitude, 1e-12))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Линейный масштаб
    axes[0].plot(freqs, amplitude, lw=0.8, color=color)
    axes[0].set_xlim(0, 25)
    axes[0].set_xlabel("Частота, Гц")
    axes[0].set_ylabel("Амплитуда, рад/с")
    axes[0].set_title(f"Линейный масштаб ({display_name})")

    # Логарифмический масштаб (дБ)
    axes[1].plot(freqs, amplitude_db, lw=0.8, color="tab:purple")
    axes[1].set_xlim(0, 25)
    finite = amplitude_db[np.isfinite(amplitude_db)]
    axes[1].set_ylim(finite.min() - 5, finite.max() + 5)
    axes[1].set_xlabel("Частота, Гц")
    axes[1].set_ylabel("Амплитуда, дБ")
    axes[1].set_title(f"Логарифмический масштаб ({display_name})")

    fig.tight_layout()
    fig.savefig(PLOTS / filename)
    plt.close(fig)

### Вывод и анализ обработанных данных

In [ ]:
axes_config = [
    ("gx", "Roll (ось X)", "tab:red", "x"),
    ("gy", "Pitch (ось Y)", "tab:green", "y"),
    ("gz", "Yaw (ось Z)", "tab:blue", "z"),
]

for col_name, display_name, color, suffix in axes_config:
    print(f"\n--- Анализ {display_name} ---")
    raw_sig = df[col_name].to_numpy(dtype=np.float64)

    # Анализ
    clean_sig, bias, sigma, mask, anomalies = analyze_axis(raw_sig, t, display_name)


    print(f"Смещение нуля (bias): {bias:+.4f} рад/с")
    print(f"СКО: {sigma:.4f} рад/с")
    print(f"Аномалий (|z| > 3.5): {len(anomalies)}")

    # Построение графиков
    plot_time_domain(
        t, clean_sig, mask, sigma,
        display_name, color, f"time_{suffix}.png"
    )
    plot_spectrum(clean_sig, fs, display_name, color, f"spectrum_{suffix}.png")
    